In [ ]:
# Clone the repo
!git clone https://github.com/Rohitxcd/bbb.git
%cd bbb

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Load dataset
url = "https://raw.githubusercontent.com/Rohitxcd/bbb/refs/heads/main/data/B3DB_classification.tsv"
df = pd.read_csv(url, sep='\t')

# Basic info
print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nLabel Distribution:")
print(df['BBB+/BBB-'].value_counts())

print("\nMissing values:")
print(df.isnull().sum())

In [ ]:
# Keep only relevant columns
df = df[['SMILES', 'BBB+/BBB-']].copy()

# Rename for convenience
df.columns = ['smiles', 'label']

# Convert labels to binary
df['label'] = df['label'].map({'BBB+': 1, 'BBB-': 0})

print(df.head())
print("\nShape:", df.shape)
print("\nLabel counts:")
print(df['label'].value_counts())

# Visualize
sns.countplot(x='label', data=df, palette=['#e74c3c', '#2ecc71'])
plt.xticks([0, 1], ['BBB- (0)', 'BBB+ (1)'])
plt.title('Class Distribution')
plt.xlabel('Label')
plt.ylabel('Count')
plt.show()

In [ ]:
# Install required libraries
!pip install rdkit transformers torch -q

from rdkit import Chem

def validate_smiles(smi):
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        return None
    return Chem.MolToSmiles(mol)  # canonicalize

df['smiles'] = df['smiles'].apply(validate_smiles)

# Drop invalid SMILES
before = len(df)
df = df.dropna(subset=['smiles']).reset_index(drop=True)
after = len(df)

print(f"Removed {before - after} invalid SMILES")
print(f"Remaining: {after} compounds")

In [ ]:
from transformers import AutoTokenizer

# Load ChemBERT tokenizer
tokenizer = AutoTokenizer.from_pretrained("seyonec/ChemBERTa-zinc-base-v1")

# Test on one SMILES
sample = df['smiles'][0]
tokens = tokenizer(sample, max_length=128, truncation=True, padding='max_length', return_tensors='pt')

print("Sample SMILES:", sample[:50], "...")
print("Input IDs shape:", tokens['input_ids'].shape)
print("Tokenizer vocab size:", tokenizer.vocab_size)

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

class BBBDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_len=128):
        self.data = dataframe.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        smiles = self.data['smiles'][idx]
        label = self.data['label'][idx]

        encoding = self.tokenizer(
            smiles,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'label': torch.tensor(label, dtype=torch.long)
        }

# Split: 80% train, 10% val, 10% test
train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df['label'])

# Create datasets
train_dataset = BBBDataset(train_df, tokenizer)
val_dataset   = BBBDataset(val_df, tokenizer)
test_dataset  = BBBDataset(test_df, tokenizer)

# Create dataloaders
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=16)
test_loader  = DataLoader(test_dataset, batch_size=16)

print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")

# Verify one batch
batch = next(iter(train_loader))
print("Input IDs shape:", batch['input_ids'].shape)
print("Labels shape:", batch['label'].shape)

In [ ]:
from transformers import AutoModelForSequenceClassification
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using: {device}")

# Load model with classification head
model = AutoModelForSequenceClassification.from_pretrained(
    "seyonec/ChemBERTa-zinc-base-v1",
    num_labels=2
)
model.to(device)

# Optimizer & scheduler
epochs = 5
optimizer = AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
total_steps = len(train_loader) * epochs
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * total_steps),
    num_training_steps=total_steps
)

print(f"Model loaded. Training for {epochs} epochs.")
print(f"Total steps: {total_steps}")

In [ ]:
from sklearn.metrics import accuracy_score, f1_score

def train_epoch(model, loader, optimizer, scheduler):
    model.train()
    total_loss = 0
    for batch in loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()

    return total_loss / len(loader)

def eval_epoch(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            preds = torch.argmax(outputs.logits, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    return all_preds, all_labels

# Train
print("Starting training...\n")
for epoch in range(epochs):
    train_loss = train_epoch(model, train_loader, optimizer, scheduler)
    preds, labels = eval_epoch(model, val_loader)

    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds)
    print(f"Epoch {epoch+1}/{epochs} | Loss: {train_loss:.4f} | Val Acc: {acc:.4f} | F1: {f1:.4f}")

In [ ]:
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix, ConfusionMatrixDisplay

# Evaluate on test set
preds, labels = eval_epoch(model, test_loader)

print("=" * 50)
print("FINAL TEST SET RESULTS")
print("=" * 50)
print(classification_report(labels, preds, target_names=['BBB- (0)', 'BBB+ (1)']))
print(f"ROC-AUC Score: {roc_auc_score(labels, preds):.4f}")

# Confusion matrix
cm = confusion_matrix(labels, preds)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['BBB-', 'BBB+'])
disp.plot(cmap='Blues')
plt.title('Confusion Matrix — Test Set')
plt.show()

In [ ]:
import os

# Save model and tokenizer
save_path = '/content/bbb/model/'
os.makedirs(save_path, exist_ok=True)

model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

print("Model saved.")

# Save the notebook
!cp /content/bbb_main.ipynb /content/bbb/notebooks/ 2>/dev/null || echo "Rename your notebook to bbb_main first if not done"

In [ ]:
%cd /content/bbb

!git config --global user.email "rohitsaharay060@gmail.com"
!git config --global user.name "Rohitxcd"

!git add .
!git commit -m "add trained ChemBERT model and notebook"
!git push origin main

In [ ]:
%cd /content/bbb

import getpass
token = getpass.getpass("Enter token: ")
!git remote set-url origin https://{token}@github.com/Rohitxcd/bbb.git
!git push origin main

In [ ]:
!git pull origin main --rebase
!git push origin main

In [ ]:
# Remove model from git tracking
!echo "model/" >> .gitignore
!git rm -r --cached model/
!git add .gitignore
!git commit -m "ignore large model files"
!git push origin main

In [ ]:
!git filter-branch --force --index-filter \
  "git rm --cached --ignore-unmatch model/model.safetensors" \
  --prune-empty --tag-name-filter cat -- --all

!git push origin main --force

In [ ]:
!cp /content/bbb_main.ipynb /content/bbb/notebooks/

%cd /content/bbb
!git add notebooks/bbb_main.ipynb
!git commit -m "add main notebook"
!git push origin main